# Electronic Substituent Effects in Molecular Embeddings

## Setup

In [ ]:
from setup import *

from joblib import Parallel, delayed
from matplotlib.lines import Line2D
from scipy.stats import pearsonr
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.pipeline import make_pipeline

## Compute Hammett Readouts

In [ ]:
n_confs = 8 # number of conformers per unique molecule, averaged
model_paths = {name: CHECKPOINTS[name] for name in ("nopre", "NT", "ANT")}
representations = ["Pooled AIMNet2"] + list(model_paths)
aimnet2_elements = {1, 5, 6, 7, 8, 9, 14, 15, 16, 17, 33, 34, 35, 53}
n_groups = 10  # substituents are split into this many groups; the probe holds out one group at a time
positions = ["meta", "para"]
canonical = lambda smiles: Chem.MolToSmiles(Chem.MolFromSmiles(smiles))
benzoic_acid = canonical("OC(=O)c1ccccc1")
axis_ends = {"meta": {"donor": canonical("Nc1cccc(C(=O)O)c1"), "acceptor": canonical("N#Cc1cccc(C(=O)O)c1")},  # 3-amino, 3-cyano
             "para": {"donor": canonical("Nc1ccc(C(=O)O)cc1"), "acceptor": canonical("N#Cc1ccc(C(=O)O)cc1")}}  # 4-amino, 4-cyano

# ---- the benzoic acids and their σ ----
def is_usable(smiles):
    """Neutral, only elements AIMNet2 was trained on, and MMFF parameters (needed for the conformers)."""
    mol = Chem.AddHs(Chem.MolFromSmiles(smiles))
    elements = {atom.GetAtomicNum() for atom in mol.GetAtoms()}
    return Chem.GetFormalCharge(mol) == 0 and elements <= aimnet2_elements and AllChem.MMFFHasAllMoleculeParams(mol)

edits = pd.read_csv(HAMMETT_PATH)
edits = edits[edits.Type.isin(["m", "p"]) & [is_usable(smiles) for smiles in edits.SMILES]].reset_index(drop=True)
edits["position"] = edits.Type.map({"m": "meta", "p": "para"})
edits["substituent"] = edits.ID.str[1:]  # ID A<n> is the para and B<n> the meta benzoic acid of substituent n
edits["smiles"] = [canonical(smiles) for smiles in edits.SMILES]

# ---- conformers and embeddings ----
# Each molecule gets n_confs conformers, and its embedding is the re-normalized mean over them (utils.encode_molecules_with_models).
def embed_conformers(smiles, n=n_confs, seed=0):
    """RDKit molecule with n MMFF-optimized ETKDG conformers (random-coordinate embedding as a fallback), or None."""
    mol = Chem.AddHs(Chem.MolFromSmiles(smiles))
    for random_coords in (False, True):
        if len(AllChem.EmbedMultipleConfs(mol, numConfs=n, randomSeed=seed, numThreads=0, useRandomCoords=random_coords)) == n:
            AllChem.MMFFOptimizeMoleculeConfs(mol, maxIters=2000, numThreads=0)
            return mol
    return None

molecules = [embed_conformers(smiles) for smiles in tqdm(edits.smiles, desc="conformers")]
embedded = [mol is not None for mol in molecules]
edits = edits[embedded].reset_index(drop=True)
molecules = [mol for mol in molecules if mol is not None]
hammett_embeddings = encode_molecules_with_models(model_paths, [embed_conformers(benzoic_acid)] + molecules, DEVICE)  # row 0 is benzoic acid
edit_vectors = {}  # representation -> [n_edits, D] array of Δh
for rep in representations:
    edit_vectors[rep] = (hammett_embeddings[rep][1:] - hammett_embeddings[rep][0]).astype(float)
sigma = edits.Sigma.to_numpy()
print(f"{len(edits)} benzoic acids ({edits.substituent.nunique()} substituents, meta and para)")

# ---- (1) linear projection on the NH2 -> CN axis (no fitting) ----
row_of = {smiles: i for i, smiles in enumerate(edits.smiles)}

def projected_sigma(rep):
    """σ̂ for every edit, with no fitting. At each position, the NH2 edit and the CN edit define an axis; each edit vector
    is projected onto it, and the position along the axis is converted to σ by linear interpolation between the two
    reference values (the NH2 edit reads σ(NH2), the CN edit reads σ(CN))."""
    estimate = np.full(len(edits), np.nan)
    for position, rows in edits.groupby("position"):
        donor, acceptor = row_of[axis_ends[position]["donor"]], row_of[axis_ends[position]["acceptor"]]
        donor_vector, acceptor_vector = edit_vectors[rep][donor], edit_vectors[rep][acceptor]
        axis = acceptor_vector - donor_vector
        midpoint = (acceptor_vector + donor_vector) / 2
        along_axis = 2 * (edit_vectors[rep][rows.index] - midpoint) @ axis / (axis @ axis)  # -1 at NH2, +1 at CN
        sigma_donor, sigma_acceptor = sigma[donor], sigma[acceptor]
        estimate[rows.index] = (sigma_acceptor + sigma_donor) / 2 + along_axis * (sigma_acceptor - sigma_donor) / 2
    return estimate

projections = {rep: projected_sigma(rep) for rep in representations}
axis_rows = [row_of[smiles] for ends in axis_ends.values() for smiles in ends.values()]
off_axis = ~np.isin(np.arange(len(edits)), axis_rows)  # the four reference molecules define the axes, so they are left out

# ---- (2) ridge regression ----
# Input: the edit vector alone (whether X is meta or para has to come from the vector itself). The substituents are
# split at random into n_groups groups; each group in turn is predicted by a probe fitted on the other groups, so every
# edit gets one prediction from a probe that never saw its substituent.
shuffled = np.random.default_rng(0).permutation(sorted(edits.substituent.unique()))
group_of = {substituent: i % n_groups for i, substituent in enumerate(shuffled)}
edits["group"] = edits.substituent.map(group_of)

def ridge_predict(X, train, test):
    """Standardized ridge regression fitted on the train rows, predictions for the test rows. The penalty is chosen by
    3-fold cross-validation inside the train rows, holding out whole substituents."""
    search = GridSearchCV(make_pipeline(StandardScaler(), Ridge()), {"ridge__alpha": np.logspace(-3, 4, 15)},
                          cv=GroupKFold(3), scoring="neg_mean_absolute_error")
    search.fit(X[train], sigma[train], groups=edits.substituent.to_numpy()[train])
    return search.predict(X[test])

splits = [((edits.group != group).to_numpy(), (edits.group == group).to_numpy()) for group in range(n_groups)]
ridge_predictions = {}  # representation -> predicted σ for every edit
for rep in representations:
    X = edit_vectors[rep]
    predictions = Parallel(n_jobs=-1)(delayed(ridge_predict)(X, train, test) for train, test in splits)
    ridge_predictions[rep] = np.full(len(edits), np.nan)
    for (_, test), prediction in zip(splits, predictions):
        ridge_predictions[rep][test] = prediction

# ---- results: projection scored by Pearson r, ridge by out-of-sample R² ----
def r_squared(predicted, true):
    return 1 - ((predicted - true) ** 2).sum() / ((true - true.mean()) ** 2).sum()

scores = {rep: {"linear projection: Pearson r": pearsonr(projections[rep][off_axis], sigma[off_axis])[0],
                "ridge, unseen substituent: R²": r_squared(ridge_predictions[rep], sigma)} for rep in representations}

## Figure 8: Hammett Projections and Ridge Predictions

In [ ]:
position_colors = {"meta": plt.get_cmap("Spectral")(0.15), "para": plt.get_cmap("Spectral")(0.85)}
figure_rows = [{"values": projections, "keep": off_axis, "score": "linear projection: Pearson r", "label": "r",
                "ylabel": "Linear Projection on NH$_2$$\\rightarrow$CN", "tick_step": 1},
               {"values": ridge_predictions, "keep": np.ones(len(edits), bool), "score": "ridge, unseen substituent: R²", "label": "R$^2$",
                "ylabel": "Ridge Regression", "tick_step": 0.5}]

def ticks_and_limit(data_half, step):
    """Symmetric ticks every `step`, and the axis limit. The outermost tick is the first multiple of step that, plus 10%,
    covers data_half, and the axis ends 10% beyond that tick: the gap between the outermost tick and the panel edge is
    then the same fraction of the panel on every axis, so on square panels the same distance on x and y."""
    outermost = step * np.ceil(data_half / 1.1 / step)
    return np.arange(-outermost, outermost + step / 2, step), 1.1 * outermost

x_ticks, x_half = ticks_and_limit(np.abs(sigma).max(), step=0.5)
# every panel has the same x range and each row the same y range, so the axes are shared and, as in the RMSD heatmap
# figure, only the outer panels carry tick labels
fig, axes = plt.subplots(2, len(representations), figsize=(17.1 * cm, 8.9 * cm), sharex=True, sharey="row")
for row, spec in enumerate(figure_rows):
    row_values = np.concatenate([spec["values"][rep][spec["keep"]] for rep in representations])
    y_ticks, y_half = ticks_and_limit(max(np.abs(np.percentile(row_values, [1, 99])).max(), x_half), spec["tick_step"])
    for column, rep in enumerate(representations):
        ax = axes[row, column]
        ax.axhline(0, color="0.85", linewidth=0.4, zorder=0)
        ax.axvline(0, color="0.85", linewidth=0.4, zorder=0)
        ax.plot([-x_half, x_half], [-x_half, x_half], color="0.4", linewidth=0.5, linestyle=(0, (3, 2)), zorder=1)  # y = x
        for position in positions:
            shown = spec["keep"] & (edits.position == position).to_numpy()
            ax.scatter(sigma[shown], spec["values"][rep][shown], s=1.5, color=position_colors[position], linewidths=0, alpha=0.7, zorder=2)
        ax.set_xlim(-x_half, x_half)
        ax.set_xticks(x_ticks)
        ax.set_ylim(-y_half, y_half)
        ax.set_yticks(y_ticks)
        ax.set_box_aspect(1)  # square panel
        ax.text(0.04, 0.97, f"{spec['label']} = {scores[rep][spec['score']]:.2f}", transform=ax.transAxes, fontsize=4.5, va="top")
        if row == 0:
            ax.set_title(MODEL_TITLES.get(rep, rep), fontsize=6, pad=1)
        if column == 0:
            ax.set_ylabel(spec["ylabel"], fontsize=6, labelpad=2)  # one y label per row: the rows plot different readouts
        ax.tick_params(labelsize=4, length=2, width=0.4, pad=1)
        ax.spines[["top", "right"]].set_visible(False)
        ax.spines[["left", "bottom"]].set_linewidth(0.4)
fig.align_ylabels(axes[:, 0])
# edges of the panel block: equal left and right margins, so the panels are centered in the figure
left, right, top, bottom = 0.0515, 0.9485, 0.9663, 0.1124
fig.text(0.5, 0.0618, "Reference σ", fontsize=6, ha="center", va="center")  # one x label shared by all panels
handles = [Line2D([0], [0], marker="o", linestyle="None", markersize=2.5, markerfacecolor=position_colors[position], markeredgewidth=0,
                  label=f"{position.capitalize()} Substitution") for position in positions]
fig.legend(handles=handles, frameon=False, fontsize=5, loc="lower center", ncol=2, handletextpad=0.3, columnspacing=1.2)
fig.subplots_adjust(left=left, right=right, top=top, bottom=bottom, hspace=0.053, wspace=0.053)
fig.savefig(OUTPUT_DIR / "hammett_edit_vectors.png", dpi=600)
plt.show()

## Hammett Readout Scores

In [ ]:
display(pd.DataFrame(scores).T.rename(index=MODEL_TITLES).round(3))